In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.linalg import eig

class STECS:
    def __init__(self, n_delays=6, rho1=0.1, rho2=1e-5, threshold=0.01):
        """
        STECS 演算法初始化
        :param n_delays: 時間濾波器階數 N
        :param rho1: 控制通道稀疏性的正則化參數 [cite: 131]
        :param rho2: 第二階段優化的正則化參數
        :param threshold: 通道保留門檻 (預設 0.01) [cite: 193]
        """
        self.N = n_delays
        self.rho1 = rho1
        self.rho2 = rho2
        self.threshold = threshold
        self.selected_channels = None
        self.w_filters = None

    def _augment_data(self, X):
        """ 建立增廣數據矩陣 X_tilde (NC x T) [cite: 100] """
        C, T = X.shape
        X_tilde = np.zeros((self.N * C, T))
        for n in range(self.N):
            # 進行時間延遲疊加 [cite: 102]
            if n == 0:
                X_tilde[:C, :] = X
            else:
                X_tilde[n*C : (n+1)*C, n:] = X[:, :-n]
        return X_tilde

    def _get_spatial_weights(self, w, C):
        """ 根據公式 (5) 分解出空間權重 s  """
        s = np.zeros(C)
        for c in range(C):
            # 取出該通道對應的 N 個時間延遲權重
            w_c = w.reshape(self.N, C)[:, c]
            # s(c) = sign(w_1c) * ||w_c||2
            s[c] = np.sign(w_c[0]) * np.linalg.norm(w_c, 2)
        return s

    def objective_j4(self, w, R1, R2, C):
        """ 公式 (9) 的目標函數  """
        # w^T * R2 * w + 1 / (w^T * R1 * w)
        term1 = w.T @ R2 @ w
        term2 = 1.0 / (w.T @ R1 @ w + 1e-9)
        # Group sparsity: rho1 * sum(||w_c||2) [cite: 118, 129]
        w_grouped = w.reshape(self.N, C)
        l21_norm = np.sum(np.linalg.norm(w_grouped, axis=0))
        return term1 + term2 + self.rho1 * l21_norm

    def fit(self, X1_list, X2_list):
        """
        訓練 STECS 模型
        X1_list, X2_list: 包含多個 trial 的列表，每個 trial 為 (C x T)
        """
        C = X1_list[0].shape[0]

        # 1. 計算增廣協方差矩陣 R1_tilde, R2_tilde [cite: 112, 155]
        R1_tilde = np.mean([self._augment_data(x) @ self._augment_data(x).T for x in X1_list], axis=0)
        R2_tilde = np.mean([self._augment_data(x) @ self._augment_data(x).T for x in X2_list], axis=0)

        # 2. 優化 w1 與 w2 (公式 9) [cite: 156]
        # 使用隨機初始化
        initial_w = np.random.randn(self.N * C)
        res1 = minimize(self.objective_j4, initial_w, args=(R1_tilde, R2_tilde, C), method='L-BFGS-B')
        res2 = minimize(self.objective_j4, initial_w, args=(R2_tilde, R1_tilde, C), method='L-BFGS-B')

        w1, w2 = res1.x, res2.x

        # 3. 分解出空間權重並正規化 [cite: 157, 191]
        s1 = self._get_spatial_weights(w1, C)
        s2 = self._get_spatial_weights(w2, C)
        s1_norm = s1 / np.max(np.abs(s1))
        s2_norm = s2 / np.max(np.abs(s2))

        # 4. 通道選擇：保留權重超過門檻的通道 [cite: 158, 193]
        self.selected_channels = np.where((np.abs(s1_norm) > self.threshold) | (np.abs(s2_norm) > self.threshold))[0]
        K = len(self.selected_channels)

        # 5. 在選定通道上學習多個非稀疏時空濾波器 (公式 10) [cite: 138, 160]
        idx_mask = []
        for c in self.selected_channels:
            for n in range(self.N):
                idx_mask.append(c + n*C)

        R1_hat = R1_tilde[np.ix_(idx_mask, idx_mask)]
        R2_hat = R2_tilde[np.ix_(idx_mask, idx_mask)]

        # 求解廣義特徵值問題 [cite: 138, 141]
        # R1_hat * w = lambda * (R2_hat + rho2 * I) * w
        vals, vecs = eig(R1_hat, R2_hat + self.rho2 * np.eye(self.N * K))

        # 取得最大與最小特徵值對應的濾波器 (各取 m 個) [cite: 141, 142]
        sort_idx = np.argsort(vals)[::-1]
        m = 3 # 論文建議各取 3 個 [cite: 196]
        self.w_filters = vecs[:, sort_idx[:m]] # 這裡僅示範取一側

        return self.selected_channels